In [38]:
# imports

import os
import io
import sys
import shutil
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [10]:
load_dotenv(override=True)

openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

gemini=OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

llama7B = OpenAI(api_key="", base_url="http://localhost:11434/v1")

qwen2=OpenAI(api_key="", base_url="http://localhost:11434/v1")

In [11]:
models=["gpt-5","gemini-2.5-flash-lite","codellama:7b","tatkal-v1:latest"]

clients ={"gpt-5":openai,"gemini-2.5-flash-lite":gemini,"codellama:7b":llama7B,"tatkal-v1:latest":qwen2}

In [12]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': False,
 'rustc': {'path': '',
  'version': '',
  'host_triple': '',
  'release': '',
  'commit_hash': ''},
 'cargo': {'path': '', 'version': ''},
 'rustup': {'path': '',
  'version': '',
  'active_toolchain': '',
  'default_toolchain': '',
  'toolchains': [],
  'targets_installed': []},
 'rust_analyzer': {'path': ''},
 'env': {'CARGO_HOME': '/Users/siddarthalegala/.cargo',
  'RUSTUP_HOME': '/Users/siddarthalegala/.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': []}

In [13]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = openai.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You do not currently have a Rust toolchain installed. You’ll need to install one.

Simplest install (macOS Apple Silicon):
1) Install rustup
- With Homebrew:
  - brew install rustup-init
  - rustup-init -y
- Or via the official script:
  - curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

2) Load Rust into your shell PATH (for this session):
- source "$HOME/.cargo/env"

3) Verify:
- rustc --version
- cargo --version

Compile and run (fastest runtime priority, single file main.rs):
Use rustc directly with maximum speed optimizations, LTO, and CPU tuning for your Apple M4.

Python snippet values:
```python
compile_command = [
    "rustc",
    "-C", "opt-level=3",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "target-cpu=native",
    "main.rs",
    "-o", "main"
]

run_command = ["./main"]
```

Notes:
- These flags prioritize runtime performance over compile time.
- On Apple Silicon (aarch64-apple-darwin), target-cpu=native enables M4-specific optimizations when available.

In [39]:
# Find rustc - try shutil.which first, then fallback to common location
rustc_path = shutil.which("rustc")
if not rustc_path:
    # Fallback to common Rust installation location
    cargo_bin = os.path.expanduser("~/.cargo/bin/rustc")
    if os.path.exists(cargo_bin):
        rustc_path = cargo_bin
    else:
        rustc_path = "rustc"  # Last resort, will fail with clear error

compile_command = [
    rustc_path,
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "main.rs",
    "-o", "main"
]

run_command = ["./main"]

In [41]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.Note: dont add any sentences only stricktly c++ code
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [42]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [43]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [44]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [45]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [46]:

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [47]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [48]:
from styles import CSS

with gr.Blocks(css=CSS) as ui:
    with gr.Row(equal_height=True):
        with gr.Column():
            python = gr.Code(
                value=python_hard,
                language="python",
                label="Python code",
                lines = 26
            )
        with gr.Column():
            rust = gr.Code(
                value="",
                language="cpp",
                label="Rust code",
                lines = 26
            )
    with gr.Row():
            runPython = gr.Button("Run Python")
            selected_model = gr.Dropdown(
                choices=models,
                value=models[0],
                label="Model"
            )
            convert_to_rust = gr.Button(f"Convert to {language}")
            run_rust = gr.Button(f"Run {language}")
    with gr.Row(equal_height=True):
        with gr.Column():
            output_python = gr.TextArea(label=f"{language} output",lines=6)
        with gr.Column():
            output_rust = gr.TextArea(label=f"{language} output",lines=6)

    runPython.click(fn=run_python, inputs=[python], outputs=[output_python])
    convert_to_rust.click(fn=port, inputs=[selected_model, python], outputs=[rust])
    run_rust.click(fn=compile_and_run, inputs=[rust], outputs=[output_rust])

ui.launch(inbrowser=True)
        
        

/var/folders/t0/v7t1g6px5_qdwtfr81fbsrf40000gn/T/ipykernel_12011/2483134033.py:3: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS) as ui:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


ALL 4 models failed to generate the rust code with compilation error.